# Correct a prediction and differentiate through the correction
# 예측을 보정하고 보정식으로 역전파하기

An MLP predicts three concentrations whose sum differs from the known feed. Compare ordinary training, a soft sum penalty, post-training projection, and training through a projection layer. The conserved sum is 1.2 mol/L in the hand example. Equality, nonnegativity, and individual steady-state balances are separate checks.

MLP가 예측한 세 농도의 합이 알려진 유입 농도와 다릅니다. 일반 학습, 합 오차 penalty, 학습 후 투영, 투영층을 포함한 학습을 비교합니다. 손계산의 보존량은 1.2 mol/L입니다. 합 등식, 비음수성, 개별 정상상태 물질수지를 각각 확인합니다.

## Setup / 실행 준비

Extract the complete lab ZIP and run this notebook in that folder, with the scripts and supplied models beside it. Install requirements.txt into this notebook’s Python environment. The cells save new results under guided-results/.

실습 ZIP 전체를 풀고 스크립트·제공 모델과 같은 폴더에서 notebook을 실행합니다. Notebook의 Python 환경에 requirements.txt를 설치합니다. 새 결과는 guided-results/에 저장됩니다.

```text
python -m pip install -r requirements.txt
```

## 1. Spread the excess equally / 1. 초과량을 세 성분에 나누기

The raw sum is 0.70+0.45+0.25=1.40, exceeding feed by 0.20. In physical Euclidean distance, the smallest total squared correction subtracts 0.20/3 from every component. The sum then loses exactly 0.20.

원래 합은 0.70+0.45+0.25=1.40으로 유입보다 0.20 큽니다. 물리 단위의 유클리드 거리에서 수정량 제곱합을 최소화하려면 각 성분에서 0.20/3을 뺍니다. 그러면 합이 정확히 0.20 줄어듭니다.

In [1]:
from pathlib import Path
import numpy as np
import torch
import week04_lab as lab
raw = torch.tensor([[.70,.45,.25]], dtype=torch.float64, requires_grad=True)
feed = torch.tensor([[1.20]], dtype=torch.float64)
excess = raw.sum(-1, keepdim=True)-feed
corrected = lab.project(raw, feed)
print("sum excess, correction per component:", excess.item(), excess.item()/3)
print("projected:", corrected.detach().numpy())
np.testing.assert_allclose(corrected.detach(), [[19/30,23/60,11/60]])
np.testing.assert_allclose(corrected.detach().sum(1), [1.2])

sum excess, correction per component: 0.19999999999999996 0.06666666666666665
projected: [[0.63333333 0.38333333 0.18333333]]


The corrected vector is about [0.633333,0.383333,0.183333]. Equal physical corrections are generally unequal after dividing by different output scales, which is why project operates after conversion back to mol/L.

보정값은 약 [0.633333,0.383333,0.183333]입니다. 같은 물리 수정량도 성분별 scale로 나누면 달라지므로 project는 mol/L로 되돌린 뒤 적용합니다.

## 2. Trace one upstream gradient / 2. Upstream gradient 하나 따라가기

The first corrected output is rawA−(rawA+rawB+rawC−feed)/3. Its derivatives with respect to raw A/B/C are 2/3,−1/3,−1/3. Increasing B makes the common correction larger and therefore reduces the corrected A.

첫 보정 출력은 rawA−(rawA+rawB+rawC−feed)/3입니다. raw A/B/C에 대한 미분은 2/3,−1/3,−1/3입니다. B가 커지면 공통으로 빼는 양이 커져 보정된 A가 줄어듭니다.

In [2]:
corrected[0,0].backward()
print("raw gradient:", raw.grad.numpy())
np.testing.assert_allclose(raw.grad, [[2/3,-1/3,-1/3]])
checks = lab.mathematical_checks()
print("projection checks:", checks)

raw gradient: [[ 0.66666667 -0.33333333 -0.33333333]]
projection checks: {'multi_equality_feasibility': True, 'projection_idempotence': True, 'gradcheck_raw_and_rhs': True, 'analytic_jacobian': True, 'scaling_transformation': True, 'nonnegative_counterexample': [[-0.053333333333333316, 0.8266666666666667, 0.4266666666666667]], 'inconsistent_rows_detected': True, 'hand_projection': [[0.6333333333333333, 0.38333333333333336, 0.18333333333333335]]}


The gradient sums to zero: a common increase in all raw components is removed by projection. mathematical_checks also tests multiple independent equality rows, the analytic Jacobian, scaling, and inconsistent constraints.

Gradient 합은 0입니다. 세 raw 성분을 함께 늘리는 변화는 투영에서 제거됩니다. mathematical_checks는 여러 독립 등식, 해석 Jacobian, scaling, 서로 모순인 제약도 검사합니다.

## 3. Find what equality does not control / 3. 합만으로 정해지지 않는 값 확인하기

Start from [0.02,0.90,0.50]. Its excess is 0.22, so subtracting 0.22/3 makes A negative. Clamping the negative component to zero adds mass without reducing another component. This is a concrete reason to inspect more than the sum residual.

[0.02,0.90,0.50]의 합 초과량은 0.22입니다. 각 성분에서 0.22/3을 빼면 A가 음수가 됩니다. 음수를 0으로 clamp하면 다른 성분을 줄이지 않은 채 총량만 늘어납니다. 그래서 합 residual 이외의 지표도 봅니다.

In [3]:
negative = lab.project(torch.tensor([[.02,.9,.5]], dtype=torch.float64), feed)
clamped = negative.clamp(min=0)
print("projected, minimum, clamped sum:", negative.tolist(), negative.min().item(), clamped.sum().item())
assert negative.min() < 0 and clamped.sum() > feed.item()
x = torch.tensor([[330.,5.,1.2]], dtype=torch.float64)
truth = lab.reference(x)
print("reference species balances:", lab.residuals(x, truth).tolist())
print("projected species balances:", lab.residuals(x, corrected.detach()).tolist())

projected, minimum, clamped sum: [[-0.053333333333333316, 0.8266666666666667, 0.4266666666666667]] -0.053333333333333316 1.2533333333333334
reference species balances: [[0.0, 6.938893903907228e-18, 6.938893903907228e-18]]
projected species balances: [[-0.01333333333333335, 0.011666666666666665, 0.0016666666666666705]]


A becomes −0.053333 mol/L and the clamped sum becomes 1.253333 mol/L. An exact sum alone also leaves individual reaction balances unconstrained. Compare all three residuals with the reference residuals near zero.

A는 −0.053333 mol/L이고 clamp한 합은 1.253333 mol/L입니다. 합이 정확해도 개별 반응 물질수지가 정해지지는 않습니다. 성분 residual 세 개를 0에 가까운 기준 residual과 비교하세요.

## 4. Run the controlled comparison / 4. 같은 조건으로 네 방법 비교하기

Keep the 900/300/300 split, initialization, batch order, and minimum-validation-MSE selection fixed. The soft loss adds lambda times squared sum error; post-projection changes an already trained model, whereas hard training backpropagates through project. Test metrics use physical units.

900/300/300 분할, 초기화, batch 순서, 최소 validation MSE 선택을 고정합니다. Soft loss는 합 오차 제곱에 lambda를 곱해 더합니다. 학습 후 투영은 완성 모델의 출력을 바꾸고 hard 학습은 project를 거쳐 역전파합니다. Test 지표는 물리 단위로 계산합니다.

In [4]:
report = lab.run(Path("guided-results"), epochs=600, seed=404)
for mode, metrics in report["test"].items():
    print(mode, "RMSE", metrics["rmse_all_mol_L"], "sum max", metrics["sum_max_mol_L"],
          "minimum", metrics["minimum_mol_L"], "species residual RMS", metrics["species_residual_rms_mol_L_min"])
assert report["protocol"]["rows"] == [900,300,300]

mlp {"rmse_components_mol_L": [0.003866391626964196, 0.003383223161934188, 0.003703555571267982], "rmse_all_mol_L": 0.003656569751646886, "sum_mae_mol_L": 0.00303156201098934, "sum_max_mol_L": 0.011651602406398176, "sum_violations_over_1e-6": 300, "minimum_mol_L": -0.003450704236282187, "negative_entries_below_minus_1e-8": 1, "species_residual_rms_mol_L_min": 0.0023009951452411617}
soft {"rmse_components_mol_L": [0.008752957171189048, 0.008742337028625698, 0.006840042688705094], "rmse_all_mol_L": 0.008161472496758939, "sum_mae_mol_L": 0.0019526083150438079, "sum_max_mol_L": 0.006302801309471029, "sum_violations_over_1e-6": 300, "minimum_mol_L": -0.010392952627957225, "negative_entries_below_minus_1e-8": 5, "species_residual_rms_mol_L_min": 0.00635618206065518}
hard {"rmse_components_mol_L": [0.0037277569834899063, 0.0025398431709099997, 0.0027025411367085717], "rmse_all_mol_L": 0.003035935004595124, "sum_mae_mol_L": 6.365278674517564e-17, "sum_max_mol_L": 3.3306690738754696e-16, "sum_v

A method can improve the sum while leaving concentration error or species residuals. Explain each column before ranking methods; this single seed does not establish a general ranking. For a new lambda comparison use run(..., soft_lambda=value, evaluate_test=False), compare training["soft"]["validation_mse"], then evaluate test once after selection. Reconstruct one reported RMSE from saved rows as a follow-up.

합이 개선돼도 농도 오차나 성분 residual이 남을 수 있습니다. 각 열의 뜻부터 설명하고 방법을 비교하세요. Seed 하나로 일반적인 순위를 정하지 않습니다. Lambda를 새로 비교할 때 run(..., soft_lambda=value, evaluate_test=False)를 쓰고 training["soft"]["validation_mse"]로 고른 뒤 test를 한 번 평가합니다. 추가로 저장 행에서 보고된 RMSE 하나를 직접 재계산하세요.

## Report / 보고서

Explain a hand prediction, the corresponding computed intermediate values, and the final interpretation. Change one setting and compare your predicted effect with the observed result. Keep training, validation/calibration, and held-out evaluation roles separate.

손으로 예측한 값, 대응하는 중간 계산값, 최종 해석을 연결해 설명합니다. 설정 하나를 바꾸고 예상한 효과와 실제 결과를 비교합니다. Train·validation/calibration·독립 평가 데이터의 역할을 구분합니다.